# 98 — Build virtual T1 integration plan (SAFE, v4)

This notebook inventories the nodal and original Geode/streamer products and
writes an ordered integration plan. Waveform alignment and hierarchical merging
are performed in notebook 99.

Integration order:

1. T1_1m
2. T1_2m
3. nodal
4. T1_Streamer

The product plan carries the integration role, priority and default super-stack
policy for every waveform product.

## 1. Configuration

In [1]:
from pathlib import Path
import re
from collections import defaultdict, deque

import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')

MANIFEST_ROOT = PROJECT_ROOT / '96_unified_nodal_stack_manifest'
REVIEW_ROOT = PROJECT_ROOT / '97_nodal_source_cluster_review'
OUT_ROOT = PROJECT_ROOT / '98_virtual_T1_integration_plan'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

STACK_MANIFEST_PATH = MANIFEST_ROOT / '96_nodal_stack_manifest.csv'
FILE_MANIFEST_PATH = MANIFEST_ROOT / '96_nodal_stack_file_manifest.csv'
CLUSTER_PATH = REVIEW_ROOT / '97_source_clusters.csv'
MEMBERSHIP_PATH = REVIEW_ROOT / '97_source_cluster_membership.csv'
PAIR_REVIEW_PATH = REVIEW_ROOT / '97_candidate_pair_review.csv'

TARGET_LINE = 'T1'
COMPONENT = 'Z'

GEODE_DATA_ROOT = PROJECT_ROOT / 'GEODE_DATA'
PREFERRED_GEODE_DATE_DIRS = ['LBS_051826']
SEARCH_ALL_LBS_DATE_DIRS = True
preferred_roots = [GEODE_DATA_ROOT / d for d in PREFERRED_GEODE_DATE_DIRS]
other_roots = (
    sorted(GEODE_DATA_ROOT.glob('LBS_*'))
    if SEARCH_ALL_LBS_DATE_DIRS and GEODE_DATA_ROOT.exists()
    else []
)
RAW_GEODE_ROOTS = []
for root in [*preferred_roots, *other_roots]:
    if root not in RAW_GEODE_ROOTS:
        RAW_GEODE_ROOTS.append(root)
RAW_GEODE_EXTENSIONS = {'.dat'}
REQUIRE_UNIQUE_BEST_GEODE_MATCH = True
INCLUDE_RAW_GEODE = True
REQUIRE_ALL_RAW_GEODE_FILES = False

# Existing notebook-97 nodal pair solutions are retained as initial timing
# estimates. Notebook 99 re-estimates cross-system alignment directly from
# common receiver traces and may search as far as +/-0.5 s.
ACCEPT_AUTOMATIC_STATUSES = {'waveform_match_supported'}
INCLUDE_INCONCLUSIVE_PAIRS = False
PAIR_WEIGHT_COLUMN = 'median_envelope_corrcoef'
MIN_PAIR_WEIGHT = 0.05

# Fallback receiver geometry. These can be changed independently if inspection
# of the SEG-2 headers or field geometry indicates different receiver layouts.
GEODE_GEOMETRY_FALLBACKS = {
    'T1_1m': {'first_x_m': 87.0, 'dx_m': 1.0, 'reverse': False},
    'T1_2m': {'first_x_m': 87.0, 'dx_m': 1.0, 'reverse': False},
    'T1_Streamer': {'first_x_m': 87.0, 'dx_m': 1.0, 'reverse': False},
}

# Lower number means earlier incorporation into each virtual shot gather.
INTEGRATION_PRIORITY = {
    'T1_1m': 10,
    'T1_2m': 20,
    'nodal': 30,
    'T1_Streamer': 40,
}

# Defaults requested for notebook 99. They are copied into the plan for
# provenance but can still be overridden in notebook 99's configuration cell.
DEFAULT_SUPER_STACK = {
    'T1_1m': False,
    'T1_2m': True,
    'nodal': False,
    'T1_Streamer': False,
}

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 280)

print('Output:', OUT_ROOT)
print('Geode data root:', GEODE_DATA_ROOT)
print('Integration priority:', INTEGRATION_PRIORITY)
print('Default super-stack policy:', DEFAULT_SUPER_STACK)
print('Raw Geode roots (in preference order):')
for root in RAW_GEODE_ROOTS:
    print(' ', root, 'exists=', root.exists())

Output: /Volumes/tachyon/LBSSP_DATA/98_virtual_T1_integration_plan
Geode data root: /Volumes/tachyon/LBSSP_DATA/GEODE_DATA
Integration priority: {'T1_1m': 10, 'T1_2m': 20, 'nodal': 30, 'T1_Streamer': 40}
Default super-stack policy: {'T1_1m': False, 'T1_2m': True, 'nodal': False, 'T1_Streamer': False}
Raw Geode roots (in preference order):
  /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051826 exists= True
  /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051626 exists= True
  /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051726 exists= True
  /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051926 exists= True


## 2. Load notebook 96 and 97 products

In [2]:
required_paths = [
    STACK_MANIFEST_PATH,
    FILE_MANIFEST_PATH,
    CLUSTER_PATH,
    MEMBERSHIP_PATH,
    PAIR_REVIEW_PATH,
]
missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Missing prerequisite files:\n'
        + '\n'.join(f'  {path}' for path in missing)
    )

stacks = pd.read_csv(STACK_MANIFEST_PATH, low_memory=False)
files = pd.read_csv(FILE_MANIFEST_PATH, low_memory=False)
clusters = pd.read_csv(CLUSTER_PATH, low_memory=False)
membership = pd.read_csv(MEMBERSHIP_PATH, low_memory=False)
pairs = pd.read_csv(PAIR_REVIEW_PATH, low_memory=False)

for frame in [stacks, clusters, membership, pairs]:
    if 'line' in frame:
        frame['line'] = frame['line'].astype(str)

stacks['source_x_m'] = pd.to_numeric(stacks['source_x_m'], errors='coerce')
membership['source_x_m'] = pd.to_numeric(
    membership['source_x_m'], errors='coerce'
)
clusters['canonical_source_x_m'] = pd.to_numeric(
    clusters['canonical_source_x_m'], errors='coerce'
)

stacks_t1 = stacks.loc[stacks.line.eq(TARGET_LINE)].copy()
membership_t1 = membership.loc[membership.line.eq(TARGET_LINE)].copy()
clusters_t1 = clusters.loc[clusters.line.eq(TARGET_LINE)].copy()
pairs_t1 = pairs.loc[pairs.line.eq(TARGET_LINE)].copy()

print('T1 stack products:', len(stacks_t1))
print('T1 source clusters:', len(clusters_t1))
print('T1 pair reviews:', len(pairs_t1))

T1 stack products: 199
T1 source clusters: 159
T1 pair reviews: 43


## 3. Build ordered virtual-shot catalog

In [3]:
shot_catalog = (
    clusters_t1
    .sort_values(
        ['canonical_source_x_m', 'source_cluster_id'],
        kind='stable',
    )
    .reset_index(drop=True)
)

shot_catalog.insert(
    0,
    'virtual_shot_number',
    np.arange(1, len(shot_catalog) + 1, dtype=int),
)
shot_catalog.insert(
    1,
    'virtual_shot_id',
    [
        f'T1_VSHOT_{number:04d}_x{source_x:07.1f}m'
        for number, source_x in zip(
            shot_catalog.virtual_shot_number,
            shot_catalog.canonical_source_x_m,
        )
    ],
)

shot_catalog['source_x_m'] = shot_catalog['canonical_source_x_m']
shot_catalog['component'] = COMPONENT

print('Ordered virtual T1 shots:', len(shot_catalog))
display(
    shot_catalog[
        [
            'virtual_shot_number', 'virtual_shot_id',
            'source_cluster_id', 'source_x_m',
            'n_stack_products', 'canonical_stack_id',
        ]
    ].head(20)
)

Ordered virtual T1 shots: 159


,virtual_shot_number,virtual_shot_id,source_cluster_id,source_x_m,n_stack_products,canonical_stack_id
0,1,T1_VSHOT_0001_x00010.0m,T1_SRC_0001,10.0,1,NODALONLYSTACK_MAY19_010M_x0010.0m
1,2,T1_VSHOT_0002_x00036.0m,T1_SRC_0002,36.0,1,NODALONLYSTACK_MAY19_036M_x0036.0m
2,3,T1_VSHOT_0003_x00043.0m,T1_SRC_0003,43.0,1,NODALSTACK_T1_T1_2m_refraction_F3047_x0043.0m
3,4,T1_VSHOT_0004_x00047.0m,T1_SRC_0004,47.0,1,NODALSTACK_T1_T1_2m_refraction_F3048_x0047.0m
4,5,T1_VSHOT_0005_x00051.0m,T1_SRC_0005,51.0,1,NODALSTACK_T1_T1_2m_refraction_F3049_x0051.0m
5,6,T1_VSHOT_0006_x00055.0m,T1_SRC_0006,55.0,1,NODALSTACK_T1_T1_2m_refraction_F3050_x0055.0m
6,7,T1_VSHOT_0007_x00059.0m,T1_SRC_0007,59.0,1,NODALSTACK_T1_T1_2m_refraction_F3051_x0059.0m
7,8,T1_VSHOT_0008_x00063.0m,T1_SRC_0008,63.0,1,NODALSTACK_T1_T1_2m_refraction_F3052_x0063.0m
8,9,T1_VSHOT_0009_x00067.0m,T1_SRC_0009,67.0,1,NODALSTACK_T1_T1_2m_refraction_F3053_x0067.0m
9,10,T1_VSHOT_0010_x00071.0m,T1_SRC_0010,71.0,1,NODALSTACK_T1_T1_2m_refraction_F3054_x0071.0m


## 4. Resolve notebook-96 nodal stack files

In [4]:
mseed_files = files.loc[
    files.file_type.astype(str).str.lower().eq('mseed')
    & files.component.astype(str).str.upper().eq(COMPONENT)
    & files.file_exists.astype(str).str.lower().isin(
        ['true', '1', 'yes']
    )
].copy()

if mseed_files.stack_id.duplicated().any():
    duplicate_rows = mseed_files.loc[
        mseed_files.stack_id.duplicated(keep=False)
    ].sort_values('stack_id')
    display(duplicate_rows)
    raise RuntimeError(
        'More than one existing Z MiniSEED product was indexed for a stack.'
    )

stack_file_lookup = dict(
    zip(mseed_files.stack_id.astype(str), mseed_files.file_path.astype(str))
)

stack_details = stacks_t1.set_index(
    stacks_t1.stack_id.astype(str),
    drop=False,
)

nodal_product_rows = []

for member in membership_t1.itertuples(index=False):
    stack_id = str(member.stack_id)
    if stack_id not in stack_file_lookup:
        continue

    details = stack_details.loc[stack_id]
    if isinstance(details, pd.DataFrame):
        details = details.iloc[0]

    shot = shot_catalog.loc[
        shot_catalog.source_cluster_id.eq(member.source_cluster_id)
    ].iloc[0]

    nodal_product_rows.append({
        'virtual_shot_number': int(shot.virtual_shot_number),
        'virtual_shot_id': shot.virtual_shot_id,
        'source_cluster_id': member.source_cluster_id,
        'source_x_m': float(shot.source_x_m),
        'product_id': stack_id,
        'product_kind': 'nodal_stack',
        'receiver_family': 'nodal',
        'integration_role': 'nodal',
        'integration_priority': INTEGRATION_PRIORITY['nodal'],
        'super_stack_default': DEFAULT_SUPER_STACK['nodal'],
        'catalog_branch': member.catalog_branch,
        'survey': details.get('survey', np.nan),
        'priority': int(member.priority),
        'merge_stage': member.merge_stage,
        'component': COMPONENT,
        'waveform_path': stack_file_lookup[stack_id],
        'geode_file_no': pd.to_numeric(
            details.get('geode_file_no', np.nan), errors='coerce'
        ),
        'n_accepted_members': pd.to_numeric(
            details.get('n_accepted_members', np.nan), errors='coerce'
        ),
        'is_canonical_stack': bool(member.is_canonical_stack),
    })

nodal_products = pd.DataFrame(nodal_product_rows)
print('Planned nodal stack products:', len(nodal_products))

Planned nodal stack products: 199


## 5. Discover original Geode/streamer files

In [5]:
def extract_numeric_tokens(path):
    """Return standalone 3-6 digit record numbers found in a filename stem."""
    return {
        int(token)
        for token in re.findall(r'(?<!\d)(\d{3,6})(?!\d)', path.stem)
    }


def acquisition_date_dir_for_path(path):
    """Return the nearest LBS_* directory containing this raw record."""
    for parent in [path.parent, *path.parents]:
        if parent.name.startswith('LBS_'):
            return parent.name
    return None


def candidate_match_reason(path, record_number):
    """
    Rank how convincingly a .dat path represents a requested Geode record.

    Lower rank is better:
      0: filename stem is exactly the record number, e.g. 3011.dat
      1: filename stem is a zero-padded form of the record number
      2: record number is a standalone numeric token in the stem
    """
    stem = path.stem.strip()
    record_text = str(int(record_number))

    if stem == record_text:
        return 0, 'exact_filename_stem'

    if stem.lstrip('0') == record_text:
        return 1, 'zero_padded_filename_stem'

    if int(record_number) in extract_numeric_tokens(path):
        return 2, 'standalone_numeric_token'

    return None, None


preferred_date_rank = {
    dirname: rank
    for rank, dirname in enumerate(PREFERRED_GEODE_DATE_DIRS)
}

raw_candidates = []

for root_rank, root in enumerate(RAW_GEODE_ROOTS):
    if not root.exists():
        continue

    for path in root.rglob('*'):
        if not path.is_file():
            continue
        if path.suffix.lower() not in RAW_GEODE_EXTENSIONS:
            continue

        acquisition_date_dir = acquisition_date_dir_for_path(path)

        raw_candidates.append({
            'raw_path': str(path),
            'filename': path.name,
            'filename_stem': path.stem,
            'numeric_tokens': extract_numeric_tokens(path),
            'size_bytes': path.stat().st_size,
            'acquisition_date_dir': acquisition_date_dir,
            'root_rank': root_rank,
            'preferred_date_rank': preferred_date_rank.get(
                acquisition_date_dir,
                len(PREFERRED_GEODE_DATE_DIRS) + 1,
            ),
        })

print('Raw Geode/streamer candidate files:', len(raw_candidates))

if raw_candidates:
    raw_candidate_catalog = pd.DataFrame(raw_candidates).drop(
        columns=['numeric_tokens']
    )
    display(
        raw_candidate_catalog.groupby(
            'acquisition_date_dir',
            dropna=False,
        ).size().reset_index(name='n_dat_files')
    )
else:
    raw_candidate_catalog = pd.DataFrame()

geode_requirements = (
    nodal_products.loc[
        nodal_products.geode_file_no.notna(),
        [
            'virtual_shot_number', 'virtual_shot_id',
            'source_cluster_id', 'source_x_m',
            'survey', 'geode_file_no',
        ],
    ]
    .drop_duplicates()
    .copy()
)

geode_requirements['geode_file_no'] = (
    geode_requirements.geode_file_no.astype(int)
)

geode_catalog_rows = []

for requirement in geode_requirements.itertuples(index=False):
    scored_matches = []

    for candidate in raw_candidates:
        path = Path(candidate['raw_path'])
        match_rank, match_reason = candidate_match_reason(
            path,
            requirement.geode_file_no,
        )

        if match_rank is None:
            continue

        score = (
            int(match_rank),
            int(candidate['preferred_date_rank']),
            int(candidate['root_rank']),
            len(path.parts),
            len(path.name),
            str(path),
        )

        scored_matches.append({
            **candidate,
            'match_rank': int(match_rank),
            'match_reason': match_reason,
            'match_score': score,
        })

    scored_matches = sorted(
        scored_matches,
        key=lambda item: item['match_score'],
    )

    selected = None
    selected_date_dir = None
    selected_match_reason = None
    best_match_score_text = None

    if not scored_matches:
        status = 'missing'
    else:
        best = scored_matches[0]
        best_primary_score = best['match_score'][:-1]

        equally_best = [
            item
            for item in scored_matches
            if item['match_score'][:-1] == best_primary_score
        ]

        if len(equally_best) == 1:
            status = 'matched_unique'
            selected = best['raw_path']
            selected_date_dir = best['acquisition_date_dir']
            selected_match_reason = best['match_reason']
            best_match_score_text = repr(best['match_score'][:-1])
        elif REQUIRE_UNIQUE_BEST_GEODE_MATCH:
            status = 'ambiguous_best_match'
        else:
            status = 'matched_first_best'
            selected = best['raw_path']
            selected_date_dir = best['acquisition_date_dir']
            selected_match_reason = best['match_reason']
            best_match_score_text = repr(best['match_score'][:-1])

    geode_catalog_rows.append({
        'virtual_shot_number': requirement.virtual_shot_number,
        'virtual_shot_id': requirement.virtual_shot_id,
        'source_cluster_id': requirement.source_cluster_id,
        'source_x_m': requirement.source_x_m,
        'survey': requirement.survey,
        'geode_file_no': int(requirement.geode_file_no),
        'match_status': status,
        'n_candidate_files': len(scored_matches),
        'selected_raw_path': selected,
        'selected_acquisition_date_dir': selected_date_dir,
        'selected_match_reason': selected_match_reason,
        'best_match_score': best_match_score_text,
        'candidate_paths': ' | '.join(
            item['raw_path'] for item in scored_matches
        ),
        'candidate_date_dirs': ' | '.join(
            str(item['acquisition_date_dir'])
            for item in scored_matches
        ),
        'candidate_match_reasons': ' | '.join(
            str(item['match_reason'])
            for item in scored_matches
        ),
    })

geode_file_catalog = pd.DataFrame(geode_catalog_rows)

if len(geode_file_catalog):
    display(
        geode_file_catalog.groupby(
            ['survey', 'match_status'],
            dropna=False,
        ).size().reset_index(name='n_files')
    )

    matched_preview = geode_file_catalog.loc[
        geode_file_catalog.match_status.isin(
            ['matched_unique', 'matched_first_best']
        ),
        [
            'survey', 'geode_file_no',
            'selected_acquisition_date_dir',
            'selected_match_reason',
            'selected_raw_path',
        ],
    ].head(30)

    if len(matched_preview):
        display(matched_preview)

unresolved_geode = geode_file_catalog.loc[
    ~geode_file_catalog.match_status.isin(
        ['matched_unique', 'matched_first_best']
    )
].copy()

if len(unresolved_geode):
    print('Unresolved original Geode/streamer files:', len(unresolved_geode))
    display(unresolved_geode.head(50))

    if REQUIRE_ALL_RAW_GEODE_FILES:
        raise RuntimeError(
            'Some original Geode/streamer files were missing or ambiguous.'
        )

Raw Geode/streamer candidate files: 326


,acquisition_date_dir,n_dat_files
0,LBS_051626,83
1,LBS_051726,87
2,LBS_051826,86
3,LBS_051926,70


,survey,match_status,n_files
0,T1_1m_refraction,matched_unique,39
1,T1_2m_refraction,matched_unique,36
2,T1_streamer_masw,matched_unique,80


,survey,geode_file_no,selected_acquisition_date_dir,selected_match_reason,selected_raw_path
0,T1_2m_refraction,3047,LBS_051826,exact_filename_stem,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...
1,T1_2m_refraction,3048,LBS_051826,exact_filename_stem,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...
2,T1_2m_refraction,3049,LBS_051826,exact_filename_stem,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...
3,T1_2m_refraction,3050,LBS_051826,exact_filename_stem,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...
4,T1_2m_refraction,3051,LBS_051826,exact_filename_stem,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...
5,T1_2m_refraction,3052,LBS_051826,exact_filename_stem,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...
6,T1_2m_refraction,3053,LBS_051826,exact_filename_stem,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...
7,T1_2m_refraction,3054,LBS_051826,exact_filename_stem,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...
8,T1_2m_refraction,3055,LBS_051826,exact_filename_stem,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...
9,T1_2m_refraction,3056,LBS_051826,exact_filename_stem,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...


## 6. Add original Geode/streamer receiver products

In [6]:
def survey_to_integration_role(survey):
    text = str(survey).lower()
    if 'streamer' in text or 'masw' in text:
        return 'T1_Streamer'
    if '2m' in text:
        return 'T1_2m'
    if '1m' in text:
        return 'T1_1m'
    raise ValueError(f'Cannot classify original receiver survey: {survey!r}')


geode_product_rows = []

if INCLUDE_RAW_GEODE and len(geode_file_catalog):
    accepted_statuses = {'matched_unique', 'matched_first_best'}
    for row in geode_file_catalog.loc[
        geode_file_catalog.match_status.isin(accepted_statuses)
    ].itertuples(index=False):
        role = survey_to_integration_role(row.survey)
        geometry = GEODE_GEOMETRY_FALLBACKS[role]
        geode_product_rows.append({
            'virtual_shot_number': int(row.virtual_shot_number),
            'virtual_shot_id': row.virtual_shot_id,
            'source_cluster_id': row.source_cluster_id,
            'source_x_m': float(row.source_x_m),
            'product_id': f'RAW_GEODE_{row.survey}_F{int(row.geode_file_no):04d}',
            'product_kind': 'geode_raw',
            'receiver_family': 'geode',
            'integration_role': role,
            'integration_priority': INTEGRATION_PRIORITY[role],
            'super_stack_default': DEFAULT_SUPER_STACK[role],
            'catalog_branch': 'original_geode_receiver_gather',
            'survey': row.survey,
            'priority': INTEGRATION_PRIORITY[role],
            'merge_stage': role,
            'component': COMPONENT,
            'waveform_path': row.selected_raw_path,
            'acquisition_date_dir': getattr(row, 'selected_acquisition_date_dir', None),
            'raw_file_match_reason': getattr(row, 'selected_match_reason', None),
            'raw_file_match_score': getattr(row, 'best_match_score', None),
            'geode_file_no': int(row.geode_file_no),
            'n_accepted_members': 1,
            'is_canonical_stack': False,
            'receiver_first_x_m_fallback': geometry['first_x_m'],
            'receiver_dx_m_fallback': geometry['dx_m'],
            'reverse_trace_order_fallback': geometry['reverse'],
        })

geode_products = pd.DataFrame(geode_product_rows)
print('Planned original Geode/streamer products:', len(geode_products))
if len(geode_products):
    display(geode_products.groupby('integration_role').size().reset_index(name='n_products'))

Planned original Geode/streamer products: 155


,integration_role,n_products
0,T1_1m,39
1,T1_2m,36
2,T1_Streamer,80


## 7. Select pair equations and solve product shifts

In [7]:
accepted_pairs = pairs_t1.loc[
    pairs_t1.automatic_status.isin(ACCEPT_AUTOMATIC_STATUSES)
].copy()

if INCLUDE_INCONCLUSIVE_PAIRS:
    accepted_pairs = pairs_t1.loc[
        pairs_t1.automatic_status.isin(
            ACCEPT_AUTOMATIC_STATUSES
            | {'waveform_match_inconclusive'}
        )
    ].copy()

accepted_pairs['measured_lag_s'] = pd.to_numeric(
    accepted_pairs['gather_lag_s'], errors='coerce'
)
accepted_pairs['equation_weight'] = pd.to_numeric(
    accepted_pairs.get(PAIR_WEIGHT_COLUMN, 1.0),
    errors='coerce',
).fillna(1.0).clip(lower=MIN_PAIR_WEIGHT)

pair_equations = accepted_pairs[
    [
        'comparison_id', 'source_cluster_id',
        'left_stack_id', 'right_stack_id',
        'measured_lag_s', 'equation_weight',
        'automatic_status',
    ]
].copy()
pair_equations['equation'] = (
    'shift(' + pair_equations.right_stack_id.astype(str)
    + ') - shift(' + pair_equations.left_stack_id.astype(str)
    + ') = ' + pair_equations.measured_lag_s.map(
        lambda value: f'{value:.9f}'
    )
)


def solve_cluster_shifts(member_frame, equation_frame):
    member_frame = member_frame.sort_values(
        ['priority', 'source_x_m', 'stack_id'],
        kind='stable',
    ).copy()

    ids = member_frame.stack_id.astype(str).tolist()
    canonical_candidates = member_frame.loc[
        member_frame.is_canonical_stack.astype(bool)
    ]
    canonical_id = (
        str(canonical_candidates.iloc[0].stack_id)
        if len(canonical_candidates)
        else ids[0]
    )

    if len(ids) == 1:
        return pd.DataFrame([{
            'stack_id': ids[0],
            'time_shift_to_canonical_s': 0.0,
            'shift_solution_status': 'single_product',
            'shift_residual_rms_s': 0.0,
            'n_shift_equations': 0,
        }])

    index = {stack_id: i for i, stack_id in enumerate(ids)}
    rows = []
    rhs = []
    weights = []

    for equation in equation_frame.itertuples(index=False):
        left = str(equation.left_stack_id)
        right = str(equation.right_stack_id)
        if left not in index or right not in index:
            continue
        vector = np.zeros(len(ids), dtype=float)
        vector[index[right]] = 1.0
        vector[index[left]] = -1.0
        rows.append(vector)
        rhs.append(float(equation.measured_lag_s))
        weights.append(float(equation.equation_weight))

    # Canonical constraint shift(canonical) = 0.
    constraint = np.zeros(len(ids), dtype=float)
    constraint[index[canonical_id]] = 1.0
    rows.append(constraint)
    rhs.append(0.0)
    weights.append(max(weights, default=1.0) * 1000.0)

    matrix = np.vstack(rows)
    rhs = np.asarray(rhs, dtype=float)
    weights = np.sqrt(np.asarray(weights, dtype=float))

    weighted_matrix = matrix * weights[:, None]
    weighted_rhs = rhs * weights
    solution, _, rank, _ = np.linalg.lstsq(
        weighted_matrix,
        weighted_rhs,
        rcond=None,
    )

    data_matrix = matrix[:-1]
    data_rhs = rhs[:-1]
    if len(data_rhs):
        residuals = data_matrix @ solution - data_rhs
        residual_rms = float(np.sqrt(np.mean(residuals ** 2)))
    else:
        residual_rms = np.nan

    status = (
        'solved_connected'
        if rank >= len(ids)
        else 'underdetermined'
    )

    return pd.DataFrame([
        {
            'stack_id': stack_id,
            'time_shift_to_canonical_s': float(solution[index[stack_id]]),
            'shift_solution_status': status,
            'shift_residual_rms_s': residual_rms,
            'n_shift_equations': len(data_rhs),
        }
        for stack_id in ids
    ])


shift_frames = []

for cluster_id, member_frame in membership_t1.groupby(
    'source_cluster_id', sort=False
):
    equations = pair_equations.loc[
        pair_equations.source_cluster_id.eq(cluster_id)
    ]
    solved = solve_cluster_shifts(member_frame, equations)
    solved['source_cluster_id'] = cluster_id
    shift_frames.append(solved)

shift_solution = pd.concat(
    shift_frames,
    ignore_index=True,
) if shift_frames else pd.DataFrame()

display(
    shift_solution.groupby(
        ['shift_solution_status'], dropna=False
    ).size().reset_index(name='n_products')
)

,shift_solution_status,n_products
0,single_product,122
1,solved_connected,75
2,underdetermined,2


## 8. Assemble final product plan

In [8]:
product_plan = pd.concat([nodal_products, geode_products], ignore_index=True, sort=False)

product_plan = product_plan.merge(
    shift_solution[[
        'source_cluster_id', 'stack_id', 'time_shift_to_canonical_s',
        'shift_solution_status', 'shift_residual_rms_s', 'n_shift_equations',
    ]],
    left_on=['source_cluster_id', 'product_id'],
    right_on=['source_cluster_id', 'stack_id'],
    how='left',
)

# The notebook-97 solution is only an initial estimate for nodal products.
# Original Geode/streamer records start at zero here; notebook 99 directly
# correlates each lower-priority product against the gather already assembled.
product_plan['initial_time_shift_s'] = pd.to_numeric(
    product_plan['time_shift_to_canonical_s'], errors='coerce'
).fillna(0.0)
product_plan.loc[product_plan.product_kind.eq('geode_raw'), 'initial_time_shift_s'] = 0.0
product_plan['time_shift_to_canonical_s'] = product_plan['initial_time_shift_s']

# Ensure every row has the new hierarchy columns, including old nodal manifests.
product_plan['integration_role'] = product_plan['integration_role'].fillna('nodal')
product_plan['integration_priority'] = pd.to_numeric(
    product_plan['integration_priority'], errors='coerce'
).fillna(product_plan.integration_role.map(INTEGRATION_PRIORITY)).astype(int)
product_plan['super_stack_default'] = product_plan['super_stack_default'].fillna(
    product_plan.integration_role.map(DEFAULT_SUPER_STACK)
).astype(bool)

# A product no longer needs a solved notebook-97 shift to be included. Notebook
# 99 performs direct common-receiver alignment and records failures explicitly.
product_plan['include_in_virtual_shot'] = product_plan.waveform_path.notna()

# Fill geometry fields for nodal products and any older rows.
for column, default in [
    ('receiver_first_x_m_fallback', np.nan),
    ('receiver_dx_m_fallback', np.nan),
    ('reverse_trace_order_fallback', False),
]:
    if column not in product_plan:
        product_plan[column] = default

product_plan = product_plan.sort_values(
    ['virtual_shot_number', 'integration_priority', 'product_id'],
    kind='stable',
).reset_index(drop=True)

print('Planned waveform products:', len(product_plan))
print('Included products:', int(product_plan.include_in_virtual_shot.sum()))
display(product_plan.groupby(
    ['integration_role', 'product_kind', 'include_in_virtual_shot', 'super_stack_default'],
    dropna=False,
).size().reset_index(name='n_products'))

Planned waveform products: 354
Included products: 354


,integration_role,product_kind,include_in_virtual_shot,super_stack_default,n_products
0,T1_1m,geode_raw,True,False,39
1,T1_2m,geode_raw,True,True,36
2,T1_Streamer,geode_raw,True,False,80
3,nodal,nodal_stack,True,False,199


## 9. Integrity checks and export

In [9]:
issues = []

if shot_catalog.virtual_shot_number.duplicated().any():
    issues.append('Duplicate virtual_shot_number values.')

if shot_catalog.source_x_m.isna().any():
    issues.append('At least one virtual shot has no source coordinate.')

included = product_plan.loc[
    product_plan.include_in_virtual_shot
]
missing_paths = included.loc[
    ~included.waveform_path.map(lambda value: Path(str(value)).exists())
]
if len(missing_paths):
    issues.append(
        f'{len(missing_paths)} included waveform paths do not exist.'
    )

underdetermined = product_plan.loc[
    product_plan.product_kind.eq('nodal_stack')
    & product_plan.shift_solution_status.eq('underdetermined')
]
if len(underdetermined):
    issues.append(
        f'{len(underdetermined)} nodal products have underdetermined shifts.'
    )

print('Plan issues:', len(issues))
for issue in issues:
    print(' -', issue)

OUTPUTS = {
    'shots': OUT_ROOT / '98_virtual_T1_shot_catalog.csv',
    'products': OUT_ROOT / '98_virtual_T1_product_plan.csv',
    'equations': OUT_ROOT / '98_virtual_T1_pair_equations.csv',
    'shifts': OUT_ROOT / '98_virtual_T1_shift_solution.csv',
    'geode_files': OUT_ROOT / '98_virtual_T1_geode_file_catalog.csv',
    'summary': OUT_ROOT / '98_virtual_T1_plan_summary.csv',
}

shot_catalog.to_csv(OUTPUTS['shots'], index=False)
product_plan.to_csv(OUTPUTS['products'], index=False)
pair_equations.to_csv(OUTPUTS['equations'], index=False)
shift_solution.to_csv(OUTPUTS['shifts'], index=False)
geode_file_catalog.to_csv(OUTPUTS['geode_files'], index=False)

summary = pd.DataFrame([
    ('virtual_T1_shots', len(shot_catalog)),
    ('planned_products', len(product_plan)),
    ('included_products', int(product_plan.include_in_virtual_shot.sum())),
    ('nodal_stack_products', int(product_plan.product_kind.eq('nodal_stack').sum())),
    ('raw_geode_products', int(product_plan.product_kind.eq('geode_raw').sum())),
    ('included_raw_geode_products', int((product_plan.product_kind.eq('geode_raw') & product_plan.include_in_virtual_shot).sum())),
    ('raw_geode_candidate_dat_files', len(raw_candidates)),
    ('unresolved_raw_geode_files', len(unresolved_geode)),
    ('accepted_pair_equations', len(pair_equations)),
    ('T1_1m_products', int(product_plan.integration_role.eq('T1_1m').sum())),
    ('T1_2m_products', int(product_plan.integration_role.eq('T1_2m').sum())),
    ('nodal_products', int(product_plan.integration_role.eq('nodal').sum())),
    ('T1_Streamer_products', int(product_plan.integration_role.eq('T1_Streamer').sum())),
    ('plan_issue_count', len(issues)),
], columns=['metric', 'value'])
summary.to_csv(OUTPUTS['summary'], index=False)
display(summary)

print('\nWritten:')
for name, path in OUTPUTS.items():
    print(f'  {name:12s} {path}')

Plan issues: 1
 - 2 nodal products have underdetermined shifts.


,metric,value
0,virtual_T1_shots,159
1,planned_products,354
2,included_products,354
3,nodal_stack_products,199
4,raw_geode_products,155
5,included_raw_geode_products,155
6,raw_geode_candidate_dat_files,326
7,unresolved_raw_geode_files,0
8,accepted_pair_equations,42
9,T1_1m_products,39



Written:
  shots        /Volumes/tachyon/LBSSP_DATA/98_virtual_T1_integration_plan/98_virtual_T1_shot_catalog.csv
  products     /Volumes/tachyon/LBSSP_DATA/98_virtual_T1_integration_plan/98_virtual_T1_product_plan.csv
  equations    /Volumes/tachyon/LBSSP_DATA/98_virtual_T1_integration_plan/98_virtual_T1_pair_equations.csv
  shifts       /Volumes/tachyon/LBSSP_DATA/98_virtual_T1_integration_plan/98_virtual_T1_shift_solution.csv
  geode_files  /Volumes/tachyon/LBSSP_DATA/98_virtual_T1_integration_plan/98_virtual_T1_geode_file_catalog.csv
  summary      /Volumes/tachyon/LBSSP_DATA/98_virtual_T1_integration_plan/98_virtual_T1_plan_summary.csv


## 10. Next step

Notebook 99 reads the product plan, applies each product's constant shift on a
common relative-time grid, forms the receiver union, combines duplicate nodal
receivers using stack-member weights, retains Geode and nodal receiver families
separately, and writes ordered per-shot MiniSEED files plus provenance catalogs.